# Children Drawings Emotion Classification (5 classes)

Using all dataset:
- 20% validation
- 80% training


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install scikit-learn

Mounted at /content/drive


In [ ]:
import os, glob, time
import numpy as np #sampling

import hashlib

#model and training
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

from PIL import Image #open image files

#splitting and metrics
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:

# Unified class names (the labels the model will learn)
# Keep the exact order stable (used for class_to_idx)
CLASSES = [
    "تفاؤل",
    "حزن",
    "غضب",
    "خوف",
    "توتر وقلق"
]
class_to_idx = {c:i for i,c in enumerate(CLASSES)} #converts the label text to number
idx_to_class = {i:c for c,i in class_to_idx.items()} #converts the number to label text
print("class_to_idx:", class_to_idx)

# Manual mapping from folder names -> unified label
# - Keys: folder names that exist
# - Values: one of CLASSES above

MAP_MAIN = {
    "تفاؤل": "تفاؤل",
    "تفاؤل": "تفاؤل",
    "حزن": "حزن",
    "غضب": "غضب",
    "خوف": "خوف",
    "توتر وقلق": "توتر وقلق",
}

class_to_idx: {'تفاؤل': 0, 'حزن': 1, 'غضب': 2, 'خوف': 3, 'توتر وقلق': 4}


In [ ]:
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

#This is example only not correct values:
#root is content/drive/GP then it joins d which is happy for example so becomes -> content/drive/GP/happy
def list_class_folders(root):
    return sorted(
        d for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    )


#method for data building
#1- walks through each folder in a dataset root
#2- checks if folder name is in the mapping
#3-collects every image path
#4- assigns the correct numeric label (based on the folder name)
#5- stores (filepath, label_idx)

def collect_samples(root, mapping):
    """
    Collect (filepath, label_idx) using a manual mapping.
    Unmapped folders are skipped safely.
    """
    samples = []
    folders = list_class_folders(root) #calls the method above

    for folder in folders:
        if folder not in mapping:
            # Unrecognized folder => skip (prevents label noise)
            continue

        unified_label = mapping[folder]
        if unified_label not in class_to_idx:
            continue

        label_idx = class_to_idx[unified_label]

        folder_path = os.path.join(root, folder)

        files = []
        for ext in IMG_EXTS: #extension is correct (we removed it from the design constraints but to be able to handle it here)
            files.extend(glob.glob(os.path.join(folder_path, f"**/*{ext}"), recursive=True))

        files = sorted(files)  # ADDED THIS

        # Add
        for fp in files:
            samples.append((fp, label_idx))

    return samples

In [ ]:
#Prepare the training and validation
SPLIT_DIR = "/content/drive/MyDrive/DS_3_SOURCES"
TRAIN_ROOT = os.path.join(SPLIT_DIR, "train")
VAL_ROOT   = os.path.join(SPLIT_DIR, "val")

#splits based on the classes we have
train_samples = collect_samples(TRAIN_ROOT, MAP_MAIN)
val_samples   = collect_samples(VAL_ROOT,   MAP_MAIN)

print("Loaded split from folder structure")
print("Train:", len(train_samples), "Val:", len(val_samples))

from collections import Counter

def print_class_distribution(samples, name):
    counts = Counter()
    for _, y in samples:
        counts[idx_to_class[y]] += 1

    print(f"\n{name} class distribution:")
    for cls in CLASSES:
        print(f"  {cls}: {counts[cls]}")
    print(f"  TOTAL: {sum(counts.values())}")

# Print distributions
print_class_distribution(train_samples, "TRAIN")
print_class_distribution(val_samples,   "VAL")

Loaded split from folder structure
Train: 1168 Val: 299

TRAIN class distribution:
  تفاؤل: 467
  حزن: 273
  غضب: 230
  خوف: 101
  توتر وقلق: 97
  TOTAL: 1168

VAL class distribution:
  تفاؤل: 118
  حزن: 70
  غضب: 59
  خوف: 27
  توتر وقلق: 25
  TOTAL: 299


In [ ]:
# Normalization for ImageNet pretrained ConvNeXt
weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
mean, std = weights.transforms().mean, weights.transforms().std

IMG_SIZE = 224

#prepare it to use it in the next cell
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.2,0.2,0.2,0.1)], p=0.6),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomRotation(degrees=8),
    transforms.ToTensor(), #converts to PyTorch Tensor
    transforms.Normalize(mean=mean, std=std),
])

val_tfms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

In [ ]:
#when the DataLoader asks for item i,
#it loads that image file, applies transforms, returns (image_tensor, label).
class FileListDataset(Dataset):
    def __init__(self, samples, transform=None):
    #we put here transform=none as the default meaning that if transform was sent it will be used,
    #if nothing was sent in the parameters it will return the same image without transformation
        self.samples = samples  # list of (path, label)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    # idx is provided automatically by PyTorch's DataLoader to indicate which sample to load
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform: #if there is transform it will apply it -> this is what we defined in the above cell
            img = self.transform(img)
        return img, label

train_ds = FileListDataset(train_samples, transform=train_tfms)
val_ds   = FileListDataset(val_samples,   transform=val_tfms)

# -------- WeightedRandomSampler (reduces bias toward big classes) --------
train_labels = [y for _, y in train_samples]
#np.bincount counts how often each class index appears and minlength=len(CLASSES) ensures missing classes get 0
class_counts = np.bincount(train_labels, minlength=len(CLASSES))
#this is where the weight is calculated, since the count is in the denominator,
#this means the higher the class count is, the lower the weight will be
class_weights = 1.0 / np.maximum(class_counts, 1) # avoid division by 0
sample_weights = [class_weights[y] for y in train_labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights), #Higher weight -> higher chance to be drawn
    num_samples=len(sample_weights), #This keeps your epoch length unchanged
    replacement=True #the same sample can appear multiple times in one epoch, if not then rare classes would still run out quickly
)

BATCH_SIZE = 32
NUM_WORKERS = 0

#note that beacause we are using a sampler with weight we did not put shuffle=true
#the WeightedRandomSampler chooses indices randomly BUT with probabilities which is how it differs from shuffle
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print("Class counts:")
for cls, idx in class_to_idx.items():
    print(cls, class_counts[idx])

print("\nClass weights:")
for cls, idx in class_to_idx.items():
    print(cls, class_weights[idx])

Class counts:
تفاؤل 467
حزن 273
غضب 230
خوف 101
توتر وقلق 97

Class weights:
تفاؤل 0.0021413276231263384
حزن 0.003663003663003663
غضب 0.004347826086956522
خوف 0.009900990099009901
توتر وقلق 0.010309278350515464


In [ ]:
#how many output neurons the model needs
num_classes = len(CLASSES)

#loads the model
model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)

# Replace classifier head
#ConvNeXt classifier is Sequential:
# (0): LayerNorm2d(...)
# (1): Flatten(...)
# (2): Linear(in_features=768, out_features=1000)
# what I am doing now is that I want it to have 5 classes as an output,
# so I am taking the in_features, storing it in a variable, to use it down and just replace
#the out_features by the num_classes
in_features = model.classifier[2].in_features


model.classifier[2] = nn.Sequential(
    nn.Dropout(0.05), # helps overfitting because our dataset is smaller comapred to imageNet
    nn.Linear(in_features, num_classes) #here is where I replaced the out_features
)

# --------------------------------------------------------------------------
# PHASE 1: freeze backbone (features) -> train only classifier
# --------------------------------------------------------------------------
for p in model.features.parameters():
    p.requires_grad = False

for p in model.classifier.parameters():
    p.requires_grad = True

model = model.to(device)

trainable = sum(p.requires_grad for p in model.parameters())
total = sum(1 for _ in model.parameters())
print(f"Trainable params tensors: {trainable}/{total}")

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 155MB/s]


Trainable params tensors: 4/182


In [ ]:
#How do we measure error? -> criterion
#Which parameters do we update, and how? -> optimizer
#How do we adjust learning speed over time? -> scheduler

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# Only params with requires_grad=True will be optimized
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=0.01
    #the weight increases while training especially in such cases like our small dataset
    #as the weight increases it starts thinking "if I see this feature -> it MUST be this class"
    #this can increase overfitting which is why we add weight_decay
    #weight decay prevents the model from relying too heavily on any single feature by discouraging large weights
)

# Scheduler: reduces LR when val loss stops improving (good for small datasets)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

In [ ]:
# During training, MixUp is used to combine pairs of images and their labels, which encourages the model
# to learn smoother and more robust decision boundaries and reduces overfitting on a small dataset.
# In training, MixUp works by linearly blending two images and computing the loss as a weighted
# combination of their labels, allowing the model to learn from intermediate representations.
# Although some training samples have mixed labels, this affects only the learning process.
# During validation and inference, MixUp is completely disabled: each image is evaluated individually
# using its original, unmixed form and a single true label. Validation metrics such as accuracy and F1
# are therefore computed in the standard way, where exactly one class is considered correct for each image.

def mixup_data(x, y, alpha=0.1):
    """Returns mixed inputs, pairs of targets, and lambda."""
    if alpha <= 0:
        return x, y, y, 1.0

    #If alpha is small -> lam tends to be near 0 or 1 (less mixing / more like one image)
    #If alpha is larger -> lam tends to be near 0.5 (more mixing)
    lam = np.random.beta(alpha, alpha)

    batch_size = x.size(0)

    #This tells you which image each image will be mixed with
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_true = [], []
    total_loss = 0.0
    n = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        n += x.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_true.extend(y.cpu().numpy().tolist())

    avg_loss = total_loss / max(n, 1)
    acc = accuracy_score(all_true, all_preds)
    macro_f1 = f1_score(all_true, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, macro_f1, all_true, all_preds


def train_one_epoch(model, loader, optimizer, use_mixup=True, mixup_alpha=0.1):
    model.train()
    total_loss = 0.0
    n = 0

    all_preds, all_true = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)

        if use_mixup:
            x_mix, y_a, y_b, lam = mixup_data(x, y, alpha=mixup_alpha)
            logits = model(x_mix)
            loss = mixup_criterion(criterion, logits, y_a, y_b, lam)

            # for metrics, use predictions from mixed forward
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(y.cpu().numpy())  # approximate, OK for monitoring

        else:
            logits = model(x)
            loss = criterion(logits, y)

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(y.cpu().numpy())

        loss.backward()

        #It prevents the gradients from becoming too large during training.
        #Large gradients can make your optimizer take a huge step and destabilize training
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # stability
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        n += x.size(0)

    avg_loss = total_loss / max(n, 1)
    train_acc = accuracy_score(all_true, all_preds)
    train_f1 = f1_score(all_true, all_preds, average="macro", zero_division=0)

    return avg_loss, train_acc, train_f1

In [ ]:
#PHASE 1 TRAINING (using MixUp pipeline)
from copy import deepcopy
import copy
EPOCHS = 20
best_macro_f1 = 0.0
patience_es = 4        # stop if no val_loss improvement for 4 epochs
min_delta = 1e-4
best_val = float("inf")
best_state = None
bad_epochs = 0

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer,
        use_mixup=True,      # <-- MixUp ON make it false if we do not want mixup
    )

    val_loss, val_acc, val_f1, y_true, y_pred = evaluate(model, val_loader)
    val_macro_f1 = f1_score(y_true, y_pred, average="macro") #-----------#

    print(
        f"phase1 | Epoch {epoch:02d} | "
        f"Train: loss={train_loss:.4f} acc={train_acc:.3f} F1={train_f1:.3f} | "
        f"Val:   loss={val_loss:.4f} acc={val_acc:.3f} F1={val_f1:.3f}"
    )

        # Early stopping logic
    if val_macro_f1 > best_macro_f1 + min_delta:
        best_macro_f1 = val_macro_f1
        best_state = copy.deepcopy(model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience_es:
            print(f"Early stopping triggered. Best val_loss={best_macro_f1:.4f}")
            break

    scheduler.step(val_loss)

# Restore best weights
if best_state is not None:
    model.load_state_dict(best_state)

phase1 | Epoch 00 | Train: loss=1.4368 acc=0.304 F1=0.293 | Val:   loss=1.3398 acc=0.445 F1=0.409
phase1 | Epoch 01 | Train: loss=1.2608 acc=0.388 F1=0.383 | Val:   loss=1.2418 acc=0.532 F1=0.489
phase1 | Epoch 02 | Train: loss=1.1066 acc=0.424 F1=0.420 | Val:   loss=1.1978 acc=0.559 F1=0.535
phase1 | Epoch 03 | Train: loss=1.0344 acc=0.410 F1=0.409 | Val:   loss=1.1974 acc=0.548 F1=0.506
phase1 | Epoch 04 | Train: loss=1.0491 acc=0.478 F1=0.477 | Val:   loss=1.2080 acc=0.532 F1=0.495
phase1 | Epoch 05 | Train: loss=1.0442 acc=0.418 F1=0.415 | Val:   loss=1.1705 acc=0.575 F1=0.539
phase1 | Epoch 06 | Train: loss=1.0089 acc=0.436 F1=0.429 | Val:   loss=1.1768 acc=0.565 F1=0.519
phase1 | Epoch 07 | Train: loss=0.9666 acc=0.453 F1=0.452 | Val:   loss=1.2337 acc=0.528 F1=0.493
phase1 | Epoch 08 | Train: loss=1.0298 acc=0.449 F1=0.445 | Val:   loss=1.2002 acc=0.559 F1=0.510
phase1 | Epoch 09 | Train: loss=0.9727 acc=0.462 F1=0.461 | Val:   loss=1.1688 acc=0.565 F1=0.533
Early stopping trigg

In [ ]:
val_loss, val_acc, val_f1, y_true, y_pred = evaluate(model, val_loader)

print(f"FINAL | Val loss={val_loss:.4f} acc={val_acc:.3f} macroF1={val_f1:.3f}")

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    target_names=CLASSES,
    digits=3,
    zero_division=0
))

import pandas as pd

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)

print("\nConfusion Matrix (rows=true, cols=pred):")
display(cm_df)


FINAL | Val loss=1.1705 acc=0.575 macroF1=0.539

Classification report:
              precision    recall  f1-score   support

       تفاؤل      0.713     0.737     0.725       118
         حزن      0.447     0.300     0.359        70
         غضب      0.700     0.475     0.566        59
         خوف      0.328     0.741     0.455        27
   توتر وقلق      0.552     0.640     0.593        25

    accuracy                          0.575       299
   macro avg      0.548     0.579     0.539       299
weighted avg      0.600     0.575     0.572       299


Confusion Matrix (rows=true, cols=pred):


,تفاؤل,حزن,غضب,خوف,توتر وقلق
تفاؤل,87,14,5,10,2
حزن,17,21,7,16,9
غضب,12,5,28,14,0
خوف,2,3,0,20,2
توتر وقلق,4,4,0,1,16


In [ ]:
# PHASE 2: UNFREEZE LAST STAGE ONLY
# Goal:
# - Keep most of the backbone frozen (prevents overfitting)
# - Unfreeze only the final ConvNeXt stage to adapt high-level features
# - Keep the classifier trainable

# 1) Freeze everything first (safe reset)
for p in model.features.parameters():
    p.requires_grad = False

# 2) Unfreeze ONLY the last stage of ConvNeXt
# In torchvision ConvNeXt, model.features is a Sequential of stages.
# The last element is typically the last stage (highest-level features).
for p in model.features[-1].parameters():
    p.requires_grad = True
for p in model.features[-2].parameters():
    p.requires_grad = True

# 3) Keep classifier trainable
for p in model.classifier.parameters():
    p.requires_grad = True

# 4) Print trainable parameters to confirm (debug / sanity)
trainable = [n for n,p in model.named_parameters() if p.requires_grad]
print("Trainable tensors:", len(trainable))
print("Examples of trainable layers:")
for n in trainable[:15]:
    print(" ", n)

Trainable tensors: 35
Examples of trainable layers:
  features.6.0.weight
  features.6.0.bias
  features.6.1.weight
  features.6.1.bias
  features.7.0.layer_scale
  features.7.0.block.0.weight
  features.7.0.block.0.bias
  features.7.0.block.2.weight
  features.7.0.block.2.bias
  features.7.0.block.3.weight
  features.7.0.block.3.bias
  features.7.0.block.5.weight
  features.7.0.block.5.bias
  features.7.1.layer_scale
  features.7.1.block.0.weight


In [ ]:
#PHASE 2 OPTIMIZER + SCHEDULER
# IMPORTANT:
# - Use a MUCH smaller LR than Phase 1
# - Use weight decay
# - Scheduler helps reduce LR when validation stops improving

# Only update the parameters that are currently trainable
backbone_params = (
    list(model.features[-1].parameters()) +
    list(model.features[-2].parameters())
)
head_params     = list(model.classifier.parameters())     # classifier

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 3e-4, "weight_decay": 0.01},
        {"params": head_params,     "lr": 3e-3, "weight_decay": 0.01},
    ]
)



scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2)

In [ ]:
import copy
EPOCHS2 = 20
patience_es = 3
min_delta = 1e-4

best_f1 = float("-inf")
best_state = None
best_epoch = -1
bad_epochs = 0

for epoch in range(EPOCHS2):
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer,
        use_mixup=True,
        mixup_alpha=0.015,
    )

    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader)  # val_f1 = macroF1 (assuming)

    scheduler.step(val_loss)

    print(
        f"phase2 | Epoch {epoch:02d} | "
        f"Train: loss={train_loss:.4f} acc={train_acc:.3f} F1={train_f1:.3f} | "
        f"Val:   loss={val_loss:.4f} acc={val_acc:.3f} F1={val_f1:.3f}"
    )

    # Early stopping on macro F1
    if val_f1 > best_f1 + min_delta:
        best_f1 = val_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience_es:
            print(f"Early stopping triggered. Best macroF1={best_f1:.4f} at epoch {best_epoch:02d}")
            break

# Restore best weights
if best_state is not None:
    model.load_state_dict(best_state)

# Final eval uses best epoch
val_loss, val_acc, val_f1, y_true, y_pred = evaluate(model, val_loader)
print(f"FINAL (best epoch {best_epoch:02d}) | Val loss={val_loss:.4f} acc={val_acc:.3f} macroF1={val_f1:.3f}")


phase2 | Epoch 00 | Train: loss=0.3295 acc=0.495 F1=0.495 | Val:   loss=1.1652 acc=0.669 F1=0.604
phase2 | Epoch 01 | Train: loss=0.2978 acc=0.613 F1=0.612 | Val:   loss=1.1680 acc=0.672 F1=0.620
phase2 | Epoch 02 | Train: loss=0.2779 acc=0.685 F1=0.685 | Val:   loss=1.1513 acc=0.679 F1=0.625
phase2 | Epoch 03 | Train: loss=0.3040 acc=0.604 F1=0.603 | Val:   loss=1.1379 acc=0.669 F1=0.614
phase2 | Epoch 04 | Train: loss=0.2873 acc=0.615 F1=0.615 | Val:   loss=1.1302 acc=0.679 F1=0.623
phase2 | Epoch 05 | Train: loss=0.3373 acc=0.522 F1=0.522 | Val:   loss=1.1329 acc=0.672 F1=0.618
Early stopping triggered. Best macroF1=0.6245 at epoch 02
FINAL (best epoch 02) | Val loss=1.1513 acc=0.679 macroF1=0.625


In [ ]:
val_loss, val_acc, val_f1, y_true, y_pred = evaluate(model, val_loader)

print(f"FINAL | Val loss={val_loss:.4f} acc={val_acc:.3f} macroF1={val_f1:.3f}")

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    target_names=CLASSES,
    digits=3,
    zero_division=0
))

import pandas as pd

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)

print("\nConfusion Matrix (rows=true, cols=pred):")
display(cm_df)


FINAL | Val loss=1.1732 acc=0.662 macroF1=0.618

Classification report:
              precision    recall  f1-score   support

       تفاؤل      0.765     0.771     0.768       118
         حزن      0.575     0.600     0.587        70
         غضب      0.615     0.678     0.645        59
         خوف      0.440     0.407     0.423        27
   توتر وقلق      0.824     0.560     0.667        25

    accuracy                          0.662       299
   macro avg      0.644     0.603     0.618       299
weighted avg      0.667     0.662     0.662       299


Confusion Matrix (rows=true, cols=pred):


,تفاؤل,حزن,غضب,خوف,توتر وقلق
تفاؤل,91,13,8,6,0
حزن,13,42,10,3,2
غضب,8,5,40,5,1
خوف,1,8,7,11,0
توتر وقلق,6,5,0,0,14


In [ ]:
torch.save(best_state, "Convnext_Best_AllDataset.pt")
torch.save(
    best_state,
    "/content/drive/MyDrive/Best_Models/Convnext_Best_AllDataset.pt"
)
print("Saved best model to Convnext_Best_AllDataset.pt")

Saved best model to Convnext_Best_AllDataset.pt
